In [18]:
%pip install --upgrade deltalake

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install deltalake.writer

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement deltalake.writer (from versions: none)
ERROR: No matching distribution found for deltalake.writer


In [1]:

import os
print(os.environ.get("JAVA_HOME"))


C:\Program Files\Java\jre1.8.0_461


In [19]:
# Generate synthetic raw data locally with controlled edge cases.
# Usage: python scripts/generate_data.py --seed 42 --out data_raw
import sys
import os
notebook_dir = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(notebook_dir, '..')))
import argparse, os, pathlib, random
import numpy as np
from faker import Faker
from datetime import datetime, timedelta, date
from mimesis import Person, Address
import rstr
import pyarrow as pa
import pyarrow.parquet as pq

#for commerce synthetic data
from faker_commerce import Provider

#timezone check
import pytz

# random character/ digit fix
import string

# Define the correct character set
charset = string.ascii_uppercase + string.digits 

# read csv
import csv

#JSON
import json
import uuid

#xlsx
import pandas as pd

#shipment pq
import pytz

# returns delta

from deltalake.writer import SchemaMode   #from pyspark.sql import SparkSession
    #from pyspark.sql.types import StructType, StructField, LongType, StringType, TimestampType, IntegerType
    #from delta import configure_spark_with_delta_pip




ImportError: cannot import name 'SchemaMode' from 'deltalake.writer' (C:\Users\jpan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\deltalake\writer\__init__.py)

In [15]:
# Timezone check
# Define the UTC+8 timezone
utc_plus_8 = pytz.timezone('Australia/Perth')  
tz = pytz.timezone('Australia/Perth')  
# Get current date in UTC+8
now = datetime.now(utc_plus_8).date()
nowtime = datetime.now(utc_plus_8)
print("Current datetime_now = datetime.now(tz)")
time_now = datetime.now(tz)
### DIMENSION TABLES

#Dimension Keys

customer_count = 80000
product_count = 25000
store_count = 5000
supplier_count = 8000
transaction_count = 1000001
transaction_backdate = 200
event_count = 2000001
sensor_count = 1000001
exchangerates_count = 365*3
shipment_count = 1000000
returns_count = 100000
v1_returns_count = 90000
v2_returns_count = 10000


customer_ids = []
product_ids = []
store_ids = []
store_channel_map = {}# for fact table
#    store_code_map = {}
supplier_ids = []
order_ids = []
product_price_map = {} # for fact table

    
# Minimal sample generation (expand to full volumes per docs)
fake = Faker('en_AU')
fake.add_provider(Provider)

Current datetime_now = datetime.now(tz)


In [ ]:

import pandas as pd
import random
from datetime import datetime, timedelta
import pathlib
import numpy as np
from deltalake.writer import write_deltalake
import pyarrow as pa

# Manual setup
class Args:
    seed = 42
    out = 'data_raw'

args = Args()

# Ensure output directory exists
def ensure_dir(p): pathlib.Path(p).mkdir(parents=True, exist_ok=True)

# Set random seed
random.seed(args.seed)
np.random.seed(args.seed)

# Paths
out = pathlib.Path(args.out)
ensure_dir(out)
orderslines_path = out / 'orders_lines.csv'
returns_v1_path = out / 'returns_v1'
returns_v2_path = out / 'returns_v2'

# Read order_lines.csv
df_orders_lines = pd.read_csv(orderslines_path)

# Filter valid rows with qty ≥ 1
valid_order_rows = df_orders_lines[df_orders_lines['qty'].fillna(0) >= 1]

# Prebatch sampling for performance (no duplicates)
sampled_rows = valid_order_rows.sample(n=returns_count).reset_index(drop=True)
    
# prebatch for v1 and v2
v1_sampled_rows = sampled_rows.head(n=v1_returns_count).reset_index(drop=True)
v2_sampled_rows = sampled_rows.tail(len(sampled_rows) - v1_returns_count).reset_index(drop=True)

# Generate synthetic returns data
base_data = []
for i, row in enumerate(v1_sampled_rows.itertuples(index=False), start=1):
    qtyret = random.randint(1, int(row.qty))
    order_ts = pd.to_datetime(row.order_ts)
    time_now = datetime.now(tz)
    time_diff = (time_now - order_ts).total_seconds()
    random_offset = random.uniform(0, time_diff)
    return_ts = (order_ts + timedelta(seconds=random_offset)).replace(tzinfo=None)
    reason = random.choice(["damaged", "wrong item", "changed mind", "late delivery"])
    base_data.append({
        "return_id": i,
        "order_id": row.order_id,
        "product_id": row.product_id,
        "return_ts": return_ts,
        "qty": qtyret,
        "reason": reason
    })

# Convert to DataFrame and then to PyArrow Table

df_returns = pd.DataFrame(base_data).sort_values(by="return_ts").reset_index(drop=True)
table = pa.Table.from_pandas(df_returns)


# Write to Delta Lake folder
write_deltalake(str(returns_v1_path), table, mode="overwrite")


## v2


reason_map = {
    "damaged": "DMG",
    "wrong item": "WRONG_IT",
    "changed mind": "CH_MND",
    "late delivery": "LT_DELIV"
}

evolved_data = []

for i, row in enumerate(v2_sampled_rows.itertuples(index=False), start=v1_returns_count + 1):
    order_ts = pd.to_datetime(row.order_ts)
    time_diff = (time_now - order_ts).total_seconds()
    return_ts = order_ts + timedelta(seconds=random.uniform(0, time_diff))

    reason = random.choice(list(reason_map.keys()))
    reason_code = reason_map[reason]

    evolved_data.append({
        "return_id": i,
        "order_id": row.order_id,
        "product_id": row.product_id,
        "return_ts": return_ts.replace(tzinfo=None),
        "qty": random.randint(1, max(1, int(row.qty))),
        "reason": reason,
        "return_reason_code": reason_code
    })

# Convert to PyArrow Table

df_evolved = pd.DataFrame(evolved_data).sort_values(by="return_ts").reset_index(drop=True)
table_evolved = pa.Table.from_pandas(df_evolved)


# Append to Delta Lake with schema evolution
write_deltalake(str(returns_v2_path), table, mode="overwrite")



In [ ]:

import duckdb

# Connect to DuckDB
con = duckdb.connect()

# Query the Delta Lake table
query = """
SELECT * FROM delta_scan('data_raw/returns_v1')
LIMIT 20
"""

# Execute and view results
df = con.execute(query).fetchdf()
print(df)


    return_id  order_id  product_id                  return_ts  qty  \
0           1    628589        7545 2025-06-15 22:51:20.966284    2   
1           2     92683       18512 2025-06-29 22:44:16.319485    1   
2           3    806344       16289 2025-05-03 11:29:23.681315    5   
3           4    575011       12810 2025-09-30 04:36:40.131614    1   
4           5    988173      394380 2025-09-15 01:33:54.844631    2   
5           6    323230       10782 2025-05-01 06:07:18.439328    9   
6           7     71206        9594 2025-09-19 19:17:13.186539    2   
7           8    324903        8942 2025-08-29 19:37:09.216061    1   
8           9    291046         721 2025-08-10 00:37:35.888911    4   
9          10    831043       24331 2025-10-03 14:31:42.944990    2   
10         11    794559       17554 2025-06-13 11:38:53.971300    1   
11         12    952875       13926 2025-09-07 11:29:26.449695    2   
12         13    775989       12811 2025-09-09 12:36:33.661713    1   
13    

In [24]:

import duckdb

# Connect to DuckDB
con = duckdb.connect()

# Query the Delta Lake table
query = """
SELECT * FROM delta_scan('data_raw/returns_v2')
LIMIT 20
"""

# Execute and view results
df = con.execute(query).fetchdf()
print(df)


    return_id  order_id  product_id                  return_ts  qty  \
0           1    628589        7545 2025-06-15 22:51:51.308747    2   
1           2     92683       18512 2025-06-29 22:48:47.113597    1   
2           3    806344       16289 2025-05-03 11:31:09.153487    5   
3           4    575011       12810 2025-09-30 04:37:16.280907    1   
4           5    988173      394380 2025-09-15 01:44:07.929867    2   
5           6    323230       10782 2025-05-01 06:11:19.664503    9   
6           7     71206        9594 2025-09-19 19:26:18.156395    2   
7           8    324903        8942 2025-08-29 19:52:29.783772    1   
8           9    291046         721 2025-08-10 00:44:28.673228    4   
9          10    831043       24331 2025-10-03 14:51:04.214727    2   
10         11    794559       17554 2025-06-13 11:40:46.488537    1   
11         12    952875       13926 2025-09-07 11:46:34.611395    2   
12         13    775989       12811 2025-09-09 12:51:18.956431    1   
13    

In [ ]:
# Replace parse_args with manual setup
class Args:
    seed = 42
    out = 'data_raw'

args = Args()

def ensure_dir(p): pathlib.Path(p).mkdir(parents=True, exist_ok=True)

def main():

    random.seed(args.seed)
    np.random.seed(args.seed)
    out = pathlib.Path(args.out)
    ensure_dir(out)

    


    spark = SparkSession.builder \
        .appName("DeltaLakeApp") \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .getOrCreate()




 ### Returns (delta)
    returns_path = out / 'returns.delta'

   # use order_lines as a reference
    # Read into DataFrame    
    orderslines_path = out/'orders_lines.csv'
    df_orders_lines = pd.read_csv(orderslines_path)

    #Pre-filter rows with qty ≥ 1
    valid_order_rows = df_orders_lines[df_orders_lines['qty'].fillna(0) >= 1]

    # Prebatch sampling for performance (no duplicates)
    sampled_rows = valid_order_rows.sample(n=returns_count).reset_index(drop=True)
    
    # prebatch for v1 and v2
    v1_sampled_rows = sampled_rows.head(n=v1_returns_count).reset_index(drop=True)
    v2_sampled_rows = sampled_rows.tail(len(sampled_rows) - v1_returns_count).reset_index(drop=True)
  # Generate base data
    base_data = []
    for i, row in enumerate(sampled_rows.itertuples(index=False), start=1):
        # Random number returned between 1 and the qty bought
        qtyret = random.randint(1, int(row.qty))

        # Calculate the time range
        order_ts = pd.to_datetime(row.order_ts)
        time_now = datetime.now(tz)
        time_diff = (time_now- order_ts).total_seconds()

        # Generate a random offset within that range
        random_offset = random.uniform(0, time_diff)

        # Create return_ts
        return_ts = (order_ts + timedelta(seconds=random_offset)).to_pydatetime()  # return_ts cannot be timezone-aware in Spark
        # reason 
        reason = random.choice(["damaged", "wrong item", "changed mind", "late delivery"])
        
        base_data.append((
            i,  # return_id
            row.order_id,  # order_id
            row.product_id, # product_id
            return_ts, # return_ts
            qtyret,  # qty
            reason  # reason
            #,row.order_dt_month # partitioning
        ))
        
    schema_v1 = StructType([
        StructField("return_id", LongType(), False),
        StructField("order_id", LongType(), False),
        StructField("product_id", StringType(), False),
        StructField("return_ts", TimestampType(), False),
        StructField("qty", IntegerType(), False),
        StructField("reason", StringType(), False)
    ])
    # Save as Delta table
    df_returns_v1 = spark.createDataFrame(base_data, schema=schema_v1)
    df_returns_v1.write.format("delta").mode("overwrite").save(str(returns_path))

    

In [10]:
#v2 evolution
    evolved_data = []
    #map evolved schema
    reason_map = {
    "damaged": "DMG",
    "wrong item": "WRONG_IT",
    "changed mind": "CH_MND",
    "late delivery": "LT_DELIV"
    }
    for i in range(returns_count + 1, returns_count + 101):  # 100 new rows to highlight append
        row = df_orders_lines.sample(1).iloc[0]
        order_id = row['order_id']
        order_ts = row['order_ts']

        qtyret = random.randint(1, max(1, int(row['qty'])))
        time_diff = (now - order_ts).total_seconds()
        return_ts = order_ts + timedelta(seconds=random.uniform(0, time_diff))

        reason = random.choice(list(reason_map.keys()))
        reason_code = reason_map[reason]

        evolved_data.append((
            i,
            int(order_id),
            row['product_id'],
            return_ts,
            qtyret,
            reason,
            row['order_dt_month'], # partitioning
            reason_code # added evolution with reasoncode
        ))
    # Updated schema with return_reason_code
    schema_v2 = StructType([
        StructField("return_id", LongType(), False),
        StructField("order_id", LongType(), False),
        StructField("product_id", StringType(), False),
        StructField("return_ts", TimestampType(), False),
        StructField("qty", IntegerType(), False),
        StructField("reason", StringType(), False),
        StructField("return_reason_code", StringType(), True) # evolved schema
    ])

    # Save as Delta table with schema evolution
    df_returns_v2 = spark.createDataFrame(base_data, schema=schema_v2)
    df_returns_v2.write.format("delta").mode("append").option("mergeSchema", "true").save(str(returns_path))
   

IndentationError: unexpected indent (3428300806.py, line 2)

In [16]:
spark = SparkSession.builder.master("local").appName("test").getOrCreate()

KeyboardInterrupt: 

In [7]:
print("creating spark session")
spark = SparkSession.builder.master("local").appName("DeltaLakeTest").config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension").config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog").getOrCreate()
print("spark session created   ")
f_test = spark.createDataFrame([(1, "test")], ["id", "value"])

creating spark session


KeyboardInterrupt: 

In [24]:
builder = SparkSession.builder \
    .appName("DeltaLakeTest") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = SparkSession.builder \
    .appName("DeltaLakeTest") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog").getOrCreate()

# spark = configure_spark_with_delta_pip(builder).getOrCreate()

df_test = spark.createDataFrame([(1, "test")], ["id", "value"])
df_test.write.format("delta").mode("overwrite").save("data_raw/test.delta")


KeyboardInterrupt: 

In [17]:

main()


KeyboardInterrupt: 